# Standard vs. Looped Transformer — complete training and evaluation lab

In this notebook, we train **two small causal language models on exactly the same text corpus** and compare their learning curves, predictive quality, parameter count, speed, memory, and generated text.

- **Standard Transformer:** four independent Transformer blocks.
- **Looped Transformer:** one Transformer block reused four times (the same block parameters are applied repeatedly).

Both models share the same tokenizer, train/validation split, embedding width, attention-head count, context length, optimizer settings, number of block applications per forward pass, and training batches.

**Run the notebook from top to bottom.** The final sections let you compare the results and generate text from your own prompts with either model or both.

## 1. What this experiment tests — and what it does not

The experiment tests whether a weight-shared recurrent-depth model can learn a useful character-level language model with fewer unique parameters than a conventional Transformer.

| Setting | Standard | Looped |
|---|---:|---:|
| Unique Transformer blocks | 4 | 1 shared block |
| Block applications per forward pass | 4 | 4 |
| Hidden width | Same | Same |
| Training data and batches | Same | Same |
| Optimizer and update count | Same | Same |

**Important fairness note:** the models have the same number of block *applications*, but they do **not** have the same parameter count. The standard model learns four independent transformations; the looped model learns one transformation and reuses it. The parameter difference is part of what we want to measure. This is not a parameter-matched or universal architecture benchmark.

More training steps and more loop iterations are different things:

- More **training steps** update the model's weights.
- More **loop iterations** apply the learned shared block additional times during one forward pass.

The default run is intentionally small enough for experimentation. For more reliable conclusions, repeat with several random seeds and larger compute budgets.

## 2. Imports and environment

The notebook uses PyTorch for the models, NumPy/Pandas for metrics, and Matplotlib for visualizations. A CUDA GPU is recommended, but CPU and Apple Silicon MPS are supported.

If a dependency is missing, install it in a separate notebook cell, then restart the kernel before running the imports. Do not upgrade PyTorch while this kernel is running. For example:

```python
%pip install torch numpy pandas matplotlib ipywidgets
```

In [ ]:
import os
import math
import json
import time
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 3. Reproducibility and compute device

We seed Python, NumPy, and PyTorch. The data-batch sampler uses separate seeded generators so both models see the same training batches in the same order. Exact bit-for-bit reproducibility can still vary by hardware and PyTorch kernels.

In [ ]:
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif DEVICE.type == "mps":
    print("Apple Silicon MPS is available.")
else:
    print("CPU mode. Consider lowering MAX_ITERS or D_MODEL for a faster first run.")

## 4. Experiment configuration

You can change these values before running the training sections. Start with the defaults, then increase `MAX_ITERS` after the first successful run.

In [ ]:
# Data and training
BATCH_SIZE = 32
BLOCK_SIZE = 128          # context length, in characters
MAX_ITERS = 1200          # optimizer updates per model
EVAL_INTERVAL = 100
EVAL_ITERS = 20            # fixed batches used for each loss evaluation
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

# Architecture — same width and same total block applications
D_MODEL = 128
N_HEADS = 4
N_BLOCK_APPLICATIONS = 4
DROPOUT = 0.1

# Output folders
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert D_MODEL % N_HEADS == 0, "D_MODEL must be divisible by N_HEADS."
assert N_BLOCK_APPLICATIONS >= 1

print({
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
    "context_length": BLOCK_SIZE,
    "optimizer_steps_per_model": MAX_ITERS,
    "embedding_width": D_MODEL,
    "attention_heads": N_HEADS,
    "block_applications": N_BLOCK_APPLICATIONS,
})

## 5. Download and inspect the Tiny Shakespeare dataset

We use the Tiny Shakespeare text corpus and train a **character-level** language model. Each character is one token. The model learns to predict the next character, not the next word or subword.

In [ ]:
DATA_PATH = DATA_DIR / "tinyshakespeare.txt"
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not DATA_PATH.exists():
    print("Downloading Tiny Shakespeare...")
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    except Exception as exc:
        raise RuntimeError(
            "Could not download Tiny Shakespeare. Check your internet connection, "
            "or manually place the corpus at data/tinyshakespeare.txt."
        ) from exc

text = DATA_PATH.read_text(encoding="utf-8")
if len(text) < 100_000:
    raise ValueError(
        f"Dataset looks unexpectedly small ({len(text):,} characters). "
        "Check data/tinyshakespeare.txt."
    )

print(f"Dataset characters: {len(text):,}")
print("Corpus preview:\n")
print(text[:1200])

## 6. Character tokenizer and train/validation split

We reserve the final 10% of the text for validation. The vocabulary is built from the complete corpus so that validation characters are representable; in a production language-modeling pipeline, vocabulary construction and data boundaries require additional care.

In [ ]:
chars = sorted(set(text))
UNK_TOKEN = "<UNK>"
itos = [UNK_TOKEN] + chars
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
vocab_size = len(itos)


def encode(s: str):
    """Map characters to integer token IDs; unseen characters map to 0."""
    return [stoi.get(ch, 0) for ch in s]


def decode(token_ids):
    """Map token IDs back to characters."""
    return "".join(
        itos[int(i)] if 0 <= int(i) < len(itos) else "�"
        for i in token_ids
    ).replace(UNK_TOKEN, "�")

split_at = int(0.90 * len(text))
train_ids = torch.tensor(encode(text[:split_at]), dtype=torch.long, device=DEVICE)
val_ids = torch.tensor(encode(text[split_at:]), dtype=torch.long, device=DEVICE)

if len(train_ids) <= BLOCK_SIZE + 1 or len(val_ids) <= BLOCK_SIZE + 1:
    raise ValueError("A split is too small for BLOCK_SIZE. Reduce BLOCK_SIZE.")

print("Vocabulary size:", vocab_size)
print("Train characters:", len(train_ids))
print("Validation characters:", len(val_ids))
print("Example encoding for ROMEO:", encode("ROMEO:"))

## 7. Mini-batches — identical data for both models

A batch contains `BATCH_SIZE` random text windows. `x` contains the input characters; `y` contains the same windows shifted one character to the right. Thus, at each position, the target is the next character.

We pre-build fixed evaluation batches. Both models are evaluated on the exact same train and validation windows, making the loss comparison less noisy.

In [ ]:
def get_batch(split: str, generator=None):
    """Sample a batch of next-character prediction windows."""
    if split not in {"train", "val"}:
        raise ValueError("split must be 'train' or 'val'")
    source = train_ids if split == "train" else val_ids
    max_start = len(source) - BLOCK_SIZE - 1
    starts = torch.randint(
        low=0,
        high=max_start,
        size=(BATCH_SIZE,),
        generator=generator,
        device="cpu",
    ).to(DEVICE)
    offsets = torch.arange(BLOCK_SIZE, device=DEVICE)
    x = source[starts[:, None] + offsets[None, :]]
    y = source[starts[:, None] + offsets[None, :] + 1]
    return x, y


# Precompute common evaluation windows for repeatable side-by-side evaluation.
_eval_generators = {
    "train": torch.Generator(device="cpu").manual_seed(SEED + 10),
    "val": torch.Generator(device="cpu").manual_seed(SEED + 20),
}
EVAL_BATCHES = {
    split: [get_batch(split, _eval_generators[split]) for _ in range(EVAL_ITERS)]
    for split in ("train", "val")
}

xb, yb = get_batch("train", torch.Generator(device="cpu").manual_seed(SEED + 30))
print("Input batch shape:", tuple(xb.shape))
print("Target batch shape:", tuple(yb.shape))
print("Input example:\n", decode(xb[0].tolist()))
print("Target example:\n", decode(yb[0].tolist()))

## 8. Shared Transformer building blocks

Both architectures use the same block implementation:

1. Pre-layer normalization.
2. Causal multi-head self-attention.
3. Residual connection.
4. Position-wise feed-forward network (MLP).
5. A second residual connection.

The attention mask prevents a position from looking at future characters.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a causal (no-future-token) mask."""
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        # [B, T, D] -> [B, heads, T, head_dim]
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        if hasattr(F, "scaled_dot_product_attention"):
            out = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.attn_dropout if self.training else 0.0,
                is_causal=True,
            )
        else:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            causal_mask = torch.tril(
                torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool)
            )
            scores = scores.masked_fill(~causal_mask, float("-inf"))
            weights = F.softmax(scores, dim=-1)
            weights = F.dropout(weights, p=self.attn_dropout, training=self.training)
            out = weights @ v

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return self.resid_dropout(self.proj(out))


class FeedForward(nn.Module):
    """Two-layer position-wise MLP."""
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-LN Transformer block with attention, MLP, and residual paths."""
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

## 9. Model A — Standard Transformer

This model has `N_BLOCK_APPLICATIONS` separate blocks. Every block has its own learned parameters.

In [ ]:
class StandardTransformerLM(nn.Module):
    """Conventional causal LM with independently parameterized blocks."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, dropout, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape
        if seq_len > self.position_embedding.num_embeddings:
            raise ValueError("Sequence exceeds configured maximum context length")
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


# We reset the seed before each model constructor. This gives both models
# reproducible initialization, including matching initial shared components
# up to the point where their parameter layouts differ.
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
standard_model = StandardTransformerLM(
    vocab_size, D_MODEL, N_HEADS, N_BLOCK_APPLICATIONS, DROPOUT, BLOCK_SIZE
).to(DEVICE)
print(standard_model)

## 10. Model B — Looped Transformer

This model has **one** learned Transformer block. Its forward pass applies that same block `N_BLOCK_APPLICATIONS` times. The hidden state changes after every application, but the block's parameters are shared.

In [ ]:
class LoopedTransformerLM(nn.Module):
    """Causal LM that repeatedly applies one shared Transformer block."""
    def __init__(self, vocab_size, d_model, n_heads, num_loops, dropout, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.shared_block = TransformerBlock(d_model, n_heads, dropout)
        self.num_loops = num_loops
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape
        if seq_len > self.position_embedding.num_embeddings:
            raise ValueError("Sequence exceeds configured maximum context length")
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]

        # The same module instance (and same weights) is called at every iteration.
        for _ in range(self.num_loops):
            x = self.shared_block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
looped_model = LoopedTransformerLM(
    vocab_size, D_MODEL, N_HEADS, N_BLOCK_APPLICATIONS, DROPOUT, BLOCK_SIZE
).to(DEVICE)
print(looped_model)

## 11. Inspect parameters and check tensor shapes

The two models should produce logits with the same shape: `[batch, context_length, vocabulary_size]`. Their trainable parameter counts should differ because only the standard model has four independently learned blocks.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model_info = pd.DataFrame([
    {
        "model": "Standard Transformer",
        "unique_transformer_blocks": N_BLOCK_APPLICATIONS,
        "block_applications_per_forward": N_BLOCK_APPLICATIONS,
        "parameters": count_parameters(standard_model),
    },
    {
        "model": "Looped Transformer",
        "unique_transformer_blocks": 1,
        "block_applications_per_forward": N_BLOCK_APPLICATIONS,
        "parameters": count_parameters(looped_model),
    },
])
model_info["parameter_storage_fp32_MiB"] = model_info["parameters"] * 4 / (1024 ** 2)
display(model_info)

with torch.no_grad():
    test_x, test_y = get_batch("train", torch.Generator(device="cpu").manual_seed(SEED + 40))
    standard_logits, standard_loss = standard_model(test_x, test_y)
    looped_logits, looped_loss = looped_model(test_x, test_y)

print("Standard logits shape:", tuple(standard_logits.shape))
print("Looped logits shape:  ", tuple(looped_logits.shape))
print("Expected shape:        ", (BATCH_SIZE, BLOCK_SIZE, vocab_size))
print("Initial standard loss:", float(standard_loss))
print("Initial looped loss:  ", float(looped_loss))

## 12. Training and evaluation functions

We use AdamW and next-character cross-entropy. Each model gets the same number of optimizer updates and the same sampled training batches. The validation batches are fixed and shared.

`perplexity = exp(cross_entropy_loss)`. Lower loss and lower perplexity indicate better next-character predictive performance under this exact tokenizer and evaluation setup.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    results = {}
    for split, batches in EVAL_BATCHES.items():
        losses = []
        for xb, yb in batches:
            _, loss = model(xb, yb)
            losses.append(loss.item())
        results[split] = float(np.mean(losses))
    model.train()
    return results


def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def train_model(model, model_key, model_label):
    """Train one architecture using a reproducible stream of mini-batches."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    batch_generator = torch.Generator(device="cpu").manual_seed(SEED + 100)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    history = {"step": [], "train_loss": [], "val_loss": []}
    sync_device()
    start_time = time.perf_counter()

    initial = estimate_loss(model)
    history["step"].append(0)
    history["train_loss"].append(initial["train"])
    history["val_loss"].append(initial["val"])
    print(f"{model_label} | step     0 | train {initial['train']:.4f} | val {initial['val']:.4f}")

    for step in range(1, MAX_ITERS + 1):
        model.train()
        xb, yb = get_batch("train", batch_generator)
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        if step % EVAL_INTERVAL == 0 or step == MAX_ITERS:
            metrics = estimate_loss(model)
            history["step"].append(step)
            history["train_loss"].append(metrics["train"])
            history["val_loss"].append(metrics["val"])
            print(
                f"{model_label} | step {step:5d} | "
                f"train {metrics['train']:.4f} | val {metrics['val']:.4f}"
            )

    sync_device()
    elapsed = time.perf_counter() - start_time
    peak_memory_mib = None
    if DEVICE.type == "cuda":
        peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

    history_df = pd.DataFrame(history)
    summary = {
        "model_key": model_key,
        "model_label": model_label,
        "parameters": count_parameters(model),
        "training_seconds": elapsed,
        "approx_training_characters_per_second": (
            MAX_ITERS * BATCH_SIZE * BLOCK_SIZE / max(elapsed, 1e-9)
        ),
        "final_train_loss": history["train_loss"][-1],
        "final_validation_loss": history["val_loss"][-1],
        "final_validation_perplexity": math.exp(min(history["val_loss"][-1], 20)),
        "peak_cuda_memory_MiB": peak_memory_mib,
        "device": str(DEVICE),
    }
    return history_df, summary

## 13. Train Model A — Standard Transformer

This cell trains the standard model. On CPU, reduce `MAX_ITERS`, `BATCH_SIZE`, or `D_MODEL` in the configuration section if this takes too long.

In [ ]:
standard_history, standard_summary = train_model(
    standard_model,
    model_key="standard",
    model_label="Standard Transformer",
)
display(pd.Series(standard_summary))

## 14. Train Model B — Looped Transformer

The looped model sees the same training windows in the same order, with the same number of optimizer updates. Its shared block receives gradients from all four uses in the forward computation.

In [ ]:
looped_history, looped_summary = train_model(
    looped_model,
    model_key="looped",
    model_label="Looped Transformer",
)
display(pd.Series(looped_summary))

## 15. Visualize training and validation loss

The first chart overlays both models' learning curves. The second compares the final validation losses. Look for convergence, generalization gaps, and whether one model learns more slowly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(standard_history["step"], standard_history["train_loss"], marker="o", label="Standard train")
axes[0].plot(standard_history["step"], standard_history["val_loss"], marker="o", label="Standard validation")
axes[0].plot(looped_history["step"], looped_history["train_loss"], marker="s", label="Looped train")
axes[0].plot(looped_history["step"], looped_history["val_loss"], marker="s", label="Looped validation")
axes[0].set_title("Training and validation learning curves")
axes[0].set_xlabel("Optimizer step")
axes[0].set_ylabel("Cross-entropy loss (nats / character)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

labels = ["Standard", "Looped"]
val_losses = [standard_summary["final_validation_loss"], looped_summary["final_validation_loss"]]
axes[1].bar(labels, val_losses)
axes[1].set_title("Final validation loss — lower is better")
axes[1].set_ylabel("Cross-entropy loss")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Validation loss difference (looped - standard):",
      round(looped_summary["final_validation_loss"] - standard_summary["final_validation_loss"], 4))

## 16. Text generation function

The model generates one character at a time. At each step, it predicts a probability distribution over the vocabulary and samples the next character.

- **Temperature below 1:** generally makes sampling more conservative.
- **Temperature above 1:** generally makes sampling more varied.
- **Top-k:** restricts sampling to the `k` highest-scoring characters. Set it to `None` to disable top-k filtering.

In [ ]:
@torch.no_grad()
def generate_text(model, prompt, max_new_tokens=300, temperature=0.8, top_k=40, seed=None):
    """Generate characters from a prompt using autoregressive sampling."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be non-negative")

    model.eval()
    prompt_ids = encode(prompt)
    if not prompt_ids:
        prompt_ids = [0]
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)

    generator = None
    if seed is not None:
        generator = torch.Generator(device=DEVICE).manual_seed(int(seed))

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            k = max(1, min(int(top_k), logits.size(-1)))
            values, _ = torch.topk(logits, k)
            logits[logits < values[:, [-1]]] = float("-inf")

        probabilities = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1, generator=generator)
        idx = torch.cat((idx, next_id), dim=1)

    return decode(idx[0].tolist())

## 17. Generate a sample from both models

Change the prompt and generation settings. Shakespeare-style prompts such as `ROMEO:` or `JULIET:` fit this training corpus, but you can also try your own text.

In [ ]:
PROMPT = "ROMEO:"
MAX_NEW_TOKENS = 500
TEMPERATURE = 0.8
TOP_K = 40                 # Use None to disable top-k filtering
GENERATION_SEED = 2026     # Change this number to sample a different continuation

print("=" * 80)
print("STANDARD TRANSFORMER")
print(generate_text(standard_model, PROMPT, MAX_NEW_TOKENS, TEMPERATURE, TOP_K, GENERATION_SEED))

print("\n" + "=" * 80)
print("LOOPED TRANSFORMER")
print(generate_text(looped_model, PROMPT, MAX_NEW_TOKENS, TEMPERATURE, TOP_K, GENERATION_SEED))

## 18. Experiment with your own prompts

Edit `MY_PROMPT` below and rerun the cell as many times as you like. Try the same prompt with both models. For example:

- `To be, or not to be,`
- `ROMEO:`
- `KING:`
- `First Citizen:`
- Your own sentence or dialogue opening.

In [ ]:
MY_PROMPT = "JULIET:"
MY_MAX_NEW_TOKENS = 400
MY_TEMPERATURE = 0.85
MY_TOP_K = 40                # None disables top-k filtering
MY_SEED = None               # None gives a fresh random sample; integer makes it repeatable

for model_name, chosen_model in [
    ("Standard Transformer", standard_model),
    ("Looped Transformer", looped_model),
]:
    print("\n" + "=" * 80)
    print(model_name)
    print(generate_text(
        chosen_model,
        prompt=MY_PROMPT,
        max_new_tokens=MY_MAX_NEW_TOKENS,
        temperature=MY_TEMPERATURE,
        top_k=MY_TOP_K,
        seed=MY_SEED,
    ))

## 19. Optional interactive prompt controls

If `ipywidgets` is installed and your notebook frontend supports widgets, use this cell to select a model, write a prompt, and adjust generation settings without editing code. If widgets are unavailable, use the editable prompt cell above.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    model_widget = widgets.Dropdown(
        options=[("Both models", "both"), ("Standard Transformer", "standard"), ("Looped Transformer", "looped")],
        value="both",
        description="Model:",
        layout=widgets.Layout(width="420px"),
    )
    prompt_widget = widgets.Textarea(
        value="ROMEO:",
        description="Prompt:",
        layout=widgets.Layout(width="95%", height="90px"),
    )
    length_widget = widgets.IntSlider(value=300, min=50, max=1000, step=50, description="New chars:", continuous_update=False)
    temperature_widget = widgets.FloatSlider(value=0.8, min=0.2, max=1.5, step=0.1, description="Temp:", continuous_update=False)
    topk_widget = widgets.IntSlider(value=40, min=1, max=min(100, vocab_size), step=1, description="Top-k:", continuous_update=False)
    seed_widget = widgets.IntText(value=2026, description="Seed:")
    generate_button = widgets.Button(description="Generate text", button_style="success")
    output_widget = widgets.Output()

    def _generate_from_widgets(_):
        with output_widget:
            clear_output(wait=True)
            selected = model_widget.value
            candidates = []
            if selected in ("both", "standard"):
                candidates.append(("Standard Transformer", standard_model))
            if selected in ("both", "looped"):
                candidates.append(("Looped Transformer", looped_model))
            for label, selected_model in candidates:
                print("=" * 80)
                print(label)
                print(generate_text(
                    selected_model,
                    prompt=prompt_widget.value,
                    max_new_tokens=length_widget.value,
                    temperature=temperature_widget.value,
                    top_k=topk_widget.value,
                    seed=seed_widget.value,
                ))
                print()

    generate_button.on_click(_generate_from_widgets)
    display(model_widget, prompt_widget, length_widget, temperature_widget, topk_widget, seed_widget, generate_button, output_widget)
except ImportError:
    print("ipywidgets is not installed. Use the editable prompt cell above instead.")

## 20. Benchmark inference speed and compare model metrics

We measure the time for one forward pass and a short autoregressive generation. Timing is hardware-dependent: compare the models on the same machine, device, and software environment. Autoregressive generation is especially affected by Python overhead in this small educational implementation.

In [ ]:
@torch.no_grad()
def benchmark_forward(model, warmup=5, repeats=30):
    model.eval()
    sample = torch.randint(0, vocab_size, (1, BLOCK_SIZE), device=DEVICE)
    for _ in range(warmup):
        model(sample)
    sync_device()
    start = time.perf_counter()
    for _ in range(repeats):
        model(sample)
    sync_device()
    return 1000 * (time.perf_counter() - start) / repeats


@torch.no_grad()
def benchmark_generation(model, prompt="ROMEO:", new_tokens=60):
    sync_device()
    start = time.perf_counter()
    _ = generate_text(model, prompt, max_new_tokens=new_tokens, temperature=0.8, top_k=40, seed=SEED)
    sync_device()
    elapsed = time.perf_counter() - start
    return elapsed, new_tokens / max(elapsed, 1e-9)


all_summaries = {
    "standard": standard_summary,
    "looped": looped_summary,
}
all_models = {
    "standard": standard_model,
    "looped": looped_model,
}
for key, mdl in all_models.items():
    forward_ms = benchmark_forward(mdl)
    generation_seconds, generation_chars_per_sec = benchmark_generation(mdl)
    all_summaries[key]["forward_latency_ms_batch1_context"] = forward_ms
    all_summaries[key]["generation_seconds_for_60_chars"] = generation_seconds
    all_summaries[key]["generation_chars_per_second"] = generation_chars_per_sec

comparison = pd.DataFrame(list(all_summaries.values())).set_index("model_label")
display(comparison[[
    "parameters",
    "final_train_loss",
    "final_validation_loss",
    "final_validation_perplexity",
    "training_seconds",
    "approx_training_characters_per_second",
    "forward_latency_ms_batch1_context",
    "generation_chars_per_second",
    "peak_cuda_memory_MiB",
]])

## 21. Visual comparison dashboard

The dashboard compares model size, final validation loss, training time, forward latency, and autoregressive generation throughput. Lower is better for loss and latency; higher is better for throughput. Do not compare raw throughput across different hardware.

In [ ]:
metric_specs = [
    ("parameters", "Trainable parameters (millions)", 1e6),
    ("final_validation_loss", "Final validation loss (lower is better)", 1.0),
    ("training_seconds", "Training time (seconds)", 1.0),
    ("forward_latency_ms_batch1_context", "Forward latency (ms; lower is faster)", 1.0),
    ("generation_chars_per_second", "Generation throughput (chars / sec)", 1.0),
]

fig, axes = plt.subplots(1, 5, figsize=(21, 4))
labels = ["Standard", "Looped"]
for ax, (metric, title, divisor) in zip(axes, metric_specs):
    values = [all_summaries["standard"][metric] / divisor,
              all_summaries["looped"][metric] / divisor]
    ax.bar(labels, values)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 22. Save models, metrics, and learning curves

The notebook saves checkpoints and CSV/JSON summaries into the `results/` directory in the current working folder. These files let you preserve results and compare later experiments.

In [ ]:
standard_history.to_csv(RESULTS_DIR / "standard_history.csv", index=False)
looped_history.to_csv(RESULTS_DIR / "looped_history.csv", index=False)

for key, model in all_models.items():
    torch.save({
        "model_state_dict": model.state_dict(),
        "model_key": key,
        "model_config": {
            "vocab_size": vocab_size,
            "d_model": D_MODEL,
            "n_heads": N_HEADS,
            "block_applications": N_BLOCK_APPLICATIONS,
            "dropout": DROPOUT,
            "block_size": BLOCK_SIZE,
        },
        "itos": itos,
        "stoi": stoi,
    }, RESULTS_DIR / f"{key}_model.pt")

for key, summary in all_summaries.items():
    (RESULTS_DIR / f"{key}_metrics.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )

combined = pd.DataFrame(list(all_summaries.values()))
combined.to_csv(RESULTS_DIR / "comparison_metrics.csv", index=False)

print("Saved artifacts to:", RESULTS_DIR.resolve())
for path in sorted(RESULTS_DIR.iterdir()):
    print(f" - {path.name} ({path.stat().st_size:,} bytes)")

## 23. How to interpret your results

Use your measured results rather than assuming one architecture is better.

1. Which model has lower final validation loss? By how much?
2. How many parameters does the looped model save?
3. Is the looped model faster or slower per forward pass on your hardware?
4. Does either model show a larger train/validation gap (possible overfitting)?
5. Do generated samples differ in coherence, repetition, and style?
6. Is the parameter saving worth any loss in quality or speed?

### Scientific limitations

- This is a small character-level dataset and model, not a modern subword LLM benchmark.
- The parameter counts differ by design.
- One random seed is not enough for a strong research conclusion.
- The same number of block applications does not imply identical FLOPs or wall-clock time.
- Training loss and validation loss measure next-character prediction, not reasoning ability.
- A looped model may need a training recipe designed specifically for recurrent depth.

### Strong follow-up experiments

1. Repeat both runs with at least three seeds and report mean and standard deviation.
2. Build a parameter-matched standard baseline by reducing its width or depth.
3. Compare at matched training FLOPs or wall-clock budget, not only matched optimizer steps.
4. Try different loop counts (1, 2, 4, 6) while holding the shared block architecture fixed.
5. Compare fixed-depth looping with adaptive halting, where the model learns how many recurrent steps to use.

**Takeaway:** this experiment isolates the effect of reusing a Transformer block. It will help you see the trade-off between unique parameters, recurrent computation, predictive quality, and speed.